# 1. Spark RDD

In [1]:
!pip install pyspark==3.5.3

from pyspark.sql import SparkSession

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 2.5 MB/s eta 0:00:0000:0100:03
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 1.1 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840630 sha256=df79f0ed44db908c449bc4475c52791eb3829776d65013abced0b078c605445c
  Stored in directory: /tmp/pip-ephem-wheel-cache-pgccsnor/wheels/97/f5/c0/947e2c0942b361ffe58651f36bd7f13772675b3863fd63d1b1
Successfully built pyspark


In [2]:
spark = SparkSession \
    .builder \
    .appName("test_app") \
    .getOrCreate()

In [3]:
spark

In [4]:
!hdfs dfs -mkdir -p /hadoop/hw2/task2/input
!hdfs dfs -put website_visits.csv /hadoop/hw2/task2/input/

In [58]:
# выводим только 5 элементов для каждой даты в формате (date, (count, site))
def top_5_sites_by_date(iterator):
    current_date = None
    n = 0
    for (date, count), site in iterator:
        if date != current_date:
            current_date = date
            n = 0
        if n < 5:
            yield (date, [(count, site)])
            n += 1


# после выполнения метода top_5_sites_by_date может быть так, что в нескольких партициях будет по 5 значений для одной даты
# в таком случае мы склеиваем все элементы с одинаковыми датами и выводим только первые 5
def merge_top5(sites1, sites2):
    # sites1, sites2 - массивы из 5 кортежей вида (count, site)
    combined = sites1 + sites2
    combined.sort(key=lambda t: (-t[0], t[1]))
    return combined[:5]
    

rdd = spark.sparkContext.textFile("/hadoop/hw2/task2/input/website_visits.csv")

# используются только методы Transform
counts = (
    rdd
    # преобразуем строку в массив вида [site, timestamp]
    .map(lambda line: line.strip().split(';'))
    # преобразуем массив в строке в вид ((date, site), 1) и убираем время из даты
    .map(lambda x: ((x[1][:10], x[0]), 1))
    # суммируем по ключу (date, site)
    .reduceByKey(lambda a, b: a + b)
    # преобразуем в вид ((date, counts), site)
    .map(lambda x: ((x[0][0], x[1]), x[0][1]))
    # сортируем по дате (по возрастанию), кол-ву посещений (по убыванию) и сайту (по возрастанию)
    .sortBy(lambda key: (key[0][0], -key[0][1], key[1]))
    # сохраняем только по 5 одинаковых дат в каждой партиции
    .mapPartitions(top_5_sites_by_date)
    # собираем все одинаковые даты и гарантированно осталяем по 5 сайтов для каждой даты
    .reduceByKey(merge_top5)
    # разделяем элементы массива по разным строчкам
    .flatMap(lambda x: [(x[0], sites[1], sites[0]) for sites in x[1]])
    # сортируем по дате, кол-ву посещений и сайтам
    .sortBy(lambda row: (row[0], -row[2], row[1]))
)

# Action и вывод
counts = counts.collect()
for sss in counts:
    print(sss)

('2024-05-26', 'https://gonzales-bautista.com/', 335)
('2024-05-26', 'http://smith.com/', 235)
('2024-05-26', 'https://www.smith.com/', 221)
('2024-05-26', 'http://www.smith.com/', 212)
('2024-05-26', 'https://smith.com/', 212)
('2024-05-27', 'https://gonzales-bautista.com/', 376)
('2024-05-27', 'https://www.smith.com/', 270)
('2024-05-27', 'https://smith.com/', 236)
('2024-05-27', 'http://smith.com/', 215)
('2024-05-27', 'http://www.smith.com/', 208)
('2024-05-28', 'https://gonzales-bautista.com/', 368)
('2024-05-28', 'https://smith.com/', 256)
('2024-05-28', 'https://www.smith.com/', 251)
('2024-05-28', 'http://smith.com/', 224)
('2024-05-28', 'http://www.smith.com/', 204)
('2024-05-29', 'https://gonzales-bautista.com/', 402)
('2024-05-29', 'https://www.smith.com/', 242)
('2024-05-29', 'http://www.smith.com/', 223)
('2024-05-29', 'https://smith.com/', 220)
('2024-05-29', 'http://smith.com/', 206)
('2024-05-30', 'https://gonzales-bautista.com/', 353)
('2024-05-30', 'https://smith.com/

# 2. Spark DataFrame

In [30]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

In [31]:
df = (
    spark.read
    .option("delimiter", ";")
    .csv("/hadoop/hw2/task2/input/website_visits.csv")
    .toDF("site", "timestamp")
)
df.show(5)

+--------------------+--------------------+
|                site|           timestamp|
+--------------------+--------------------+
|https://gonzales-...|2024-05-28 05:58:...|
|https://gonzales-...|2024-05-26 03:44:...|
|https://www.davis...|2024-05-31 17:22:...|
|https://www.monto...|2024-05-29 13:41:...|
|https://gonzales-...|2024-06-01 06:49:...|
+--------------------+--------------------+
only showing top 5 rows



In [59]:
# создаём окно для каждой даты, в котором все элементы отсортированы по count
window_spec = Window.partitionBy("date").orderBy(sf.desc("count"), sf.asc("site"))

counts_df = (
    df
    # убираем время из даты
    .withColumn("date", sf.substring(sf.col("timestamp"), 1, 10))
    # гурппируем по дате и сайту
    .groupBy("date", "site")
    # в новом столбце "count" пишется кол-во ключей вида (date, site)
    .count()
    # добавляем новый столбец, который нумерует строки внутри каждой даты по убыванию count
    .withColumn("number", sf.row_number().over(window_spec))
    # убираем все строки, с номером больше 5
    .filter(sf.col("number") <= 5)
    # сортируем по дате, кол-ву посещений и сайтам
    .orderBy(sf.asc("date"), sf.desc("count"), sf.asc("site"))
    # отбрасываем столбец number
    .drop("number")
)

counts_df.show(40)

+----------+--------------------+-----+
|      date|                site|count|
+----------+--------------------+-----+
|2024-05-26|https://gonzales-...|  335|
|2024-05-26|   http://smith.com/|  235|
|2024-05-26|https://www.smith...|  221|
|2024-05-26|http://www.smith....|  212|
|2024-05-26|  https://smith.com/|  212|
|2024-05-27|https://gonzales-...|  376|
|2024-05-27|https://www.smith...|  270|
|2024-05-27|  https://smith.com/|  236|
|2024-05-27|   http://smith.com/|  215|
|2024-05-27|http://www.smith....|  208|
|2024-05-28|https://gonzales-...|  368|
|2024-05-28|  https://smith.com/|  256|
|2024-05-28|https://www.smith...|  251|
|2024-05-28|   http://smith.com/|  224|
|2024-05-28|http://www.smith....|  204|
|2024-05-29|https://gonzales-...|  402|
|2024-05-29|https://www.smith...|  242|
|2024-05-29|http://www.smith....|  223|
|2024-05-29|  https://smith.com/|  220|
|2024-05-29|   http://smith.com/|  206|
|2024-05-30|https://gonzales-...|  353|
|2024-05-30|  https://smith.com/|  246|


# Проверка результатов с MapReduce

In [60]:
counts_df_formatted = [(row[0], row[1], row[2]) for row in counts_df.collect()]

In [63]:
print(counts == counts_df_formatted)

True


То есть Spark RDD и Spark DataFrame равны между собой. Теперь сравним MapReduce с одним из способов в Spark

In [84]:
map_reduce_rdd = spark.sparkContext.textFile("/hadoop/hw1/task2/mr2/output/haddop_visits_result.csv")
map_reduce_result = (
    map_reduce_rdd
    .map(lambda line: line.strip().split('\t'))
    .map(lambda x: (x[0], x[1], int(x[2])))
)
map_reduce_formatted = [(row[0], row[1], row[2]) for row in map_reduce_result.collect()]

In [85]:
print(counts == map_reduce_formatted)

True


### То есть все 3 файла равны между собой. Теперь можно гордиться собой и довольным пойти спать